In [3]:
!pip install bertopic sentence-transformers dask pyarrow

In [4]:
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 74.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 65.4 MB/s eta 0:00:00:00:010:01
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import os
import gc
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')

# Академическая цветовая палитра
ACADEMIC_PALETTE = ['#2F4F4F', '#8B0000', '#4682B4', '#CD853F', '#556B2F', '#708090', '#8B4513']

# Директория для сохранения графиков
OUTPUT_DIR = Path("/kaggle/working/plots")
OUTPUT_DIR.mkdir(exist_ok=True)

def save_plot(fig, hypothesis_name: str):
    fig.update_layout(plot_bgcolor='white', paper_bgcolor='white', font_family="Arial")
    fname = f"{hypothesis_name}.html"
    fig.write_html(OUTPUT_DIR / fname, full_html=False, include_plotlyjs="cdn")
    fig.show()

# --- загрузка данных ---
print("Настройка Kaggle API и скачивание датасета...")
os.environ['KAGGLE_USERNAME'] = "Toxele"
os.environ['KAGGLE_KEY'] = "KGAT_c9b19e623ab66fa40e3a2a02b05d649d"

data_dir = Path(".")
if not list(data_dir.rglob("data_comments_*.csv")):
    !kaggle datasets download -d lavrovalexei/my-dataset --unzip
    print("Датасет распакован.")

def load_data():
    comment_files = list(data_dir.rglob("data_comments_*.csv"))
    df_list = []
    
    for f in comment_files:
        try:
            # Читаем чанками для безопасности памяти
            temp = pd.read_csv(f, low_memory=False, on_bad_lines='skip', 
                               usecols=lambda c: c in ['text', 'publish_date', 'comment_publish_date', 
                                                       'parent_comment_id', 'like_count', 'video_id', 'author_id'])
            
            # Унификация колонок дат
            if 'comment_publish_date' in temp.columns:
                temp.rename(columns={'comment_publish_date': 'comment_date'}, inplace=True)
            elif 'publish_date' in temp.columns:
                temp.rename(columns={'publish_date': 'comment_date'}, inplace=True)
                
            temp['dataset'] = f.stem.replace("data_comments_", "")
            df_list.append(temp)
        except Exception as e:
            print(f"Ошибка чтения {f.name}: {e}")
            
    df = pd.concat(df_list, ignore_index=True)
    del df_list; gc.collect()
    
    # Базовая очистка
    df['comment_date'] = pd.to_datetime(df['comment_date'], errors='coerce')
    df = df.dropna(subset=['comment_date', 'text'])
    df['year'] = df['comment_date'].dt.year.astype(int)
    df['month_year'] = df['comment_date'].dt.to_period('M').dt.to_timestamp()
    df['text'] = df['text'].astype(str).str.lower()
    
    # Флаг: является ли комментарий ответом (корневой или ответ в ветке)
    df['is_reply'] = df['parent_comment_id'].notna().astype(int)
    
    print(f"Загружено {len(df):,} комментариев.")
    return df

df = load_data()

2026-05-15 16:10:11.993670: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778861412.216870      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778861412.282000      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778861412.822633      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778861412.822676      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778861412.822679      57 computation_placer.cc:177] computation placer alr

Настройка Kaggle API и скачивание датасета...
Dataset URL: https://www.kaggle.com/datasets/lavrovalexei/my-dataset
License(s): CC0-1.0
100%|█████████████████████████████████████████| 676M/676M [00:05<00:00, 132MB/s]

Датасет распакован.
Загружено 4,825,616 комментариев.


In [6]:
print("--- Гипотеза 1: Индекс ретроспективной актуализации ---")

# Расширенный словарь маркеров современности
modern_dict = [
    # Политические акторы и страны
    r'путин', r'байден', r'обама', r'трамп', r'зеленск', r'порошенко', r'янукович', r'чубайс', r'ельцин',
    r'сша', r'нато', r'евросоюз', r'ес\b', r'запад', r'америка', r'украин', r'росси', r'рф\b',
    # Идеологические и пропагандистские маркеры
    r'сво\b', r'спецопераци', r'денацификаци', r'демилитаризаци', r'укрофашист', r'рашист', r'ватник', r'либерал',
    r'нацик', r'бандер', r'биолаборатор', r'гойда', r'русофоб', r'скрепы', r'пятая колонна', r'иноагент',
    # Современный военный сленг и технологии
    r'бпла', r'дрон', r'герань', r'байрактар', r'хаймарс', r'himars', r'джавелин', r'тепловизор', 
    r'чвк', r'вагнер', r'ахмат', r'азов', r'лбс', r'мобилизаци', r'военкор', r'инфовойна', r'фейк'
]

# Компилируем регулярное выражение для скорости
modern_regex = '|'.join(modern_dict)

# Считаем количество вхождений маркеров в тексте
df['modern_hits'] = df['text'].str.count(modern_regex)
# Считаем общее количество слов в комментарии (приближенно по пробелам)
df['word_count'] = df['text'].str.count(r'\w+')
# Избегаем деления на ноль
df['word_count'] = df['word_count'].replace(0, 1)

# Индекс актуализации: доля современных токенов (в промилле, ‰, для наглядности графика)
df['actualization_index'] = (df['modern_hits'] / df['word_count']) * 1000

# Агрегируем по кварталам (чтобы сгладить шум по месяцам)
df['quarter_year'] = df['comment_date'].dt.to_period('Q').dt.to_timestamp()
h1_data = df[df['year'] >= 2010].groupby('quarter_year')['actualization_index'].mean().reset_index()

# Сглаживание скользящим средним (2 квартала)
h1_data['smoothed_index'] = h1_data['actualization_index'].rolling(window=2, min_periods=1).mean()

fig1 = px.line(h1_data, x='quarter_year', y='smoothed_index', 
               title="Индекс ретроспективной актуализации (2010–2025)",
               labels={'smoothed_index': 'Плотность маркеров (‰)', 'quarter_year': 'Год'},
               color_discrete_sequence=[ACADEMIC_PALETTE[0]])

# Метки исторических событий
fig1.add_vline(x="2014-03-01", line_dash="dash", line_color=ACADEMIC_PALETTE[1])
fig1.add_annotation(x="2014-03-01", y=0.9, yref="paper", text="Крым / Начало конфликта", 
                    showarrow=False, font=dict(color=ACADEMIC_PALETTE[1], size=12), xanchor="right")

fig1.add_vline(x="2022-02-24", line_dash="dash", line_color=ACADEMIC_PALETTE[1])
fig1.add_annotation(x="2022-02-24", y=0.9, yref="paper", text="Начало СВО", 
                    showarrow=False, font=dict(color=ACADEMIC_PALETTE[1], size=12), xanchor="left")

save_plot(fig1, "01_retrospective_actualization")

--- Гипотеза 1: Индекс ретроспективной актуализации ---


In [7]:
print("--- Гипотеза 2: Семантическая структура памяти (BERTopic) ---")

# Берем датасет Ленинграда
leningrad_df = df[df['dataset'] == 'leningrad'].copy()

if len(leningrad_df) > 0:
    # Стратифицированное сэмплирование (по годам), чтобы не исказить эпохи. Объем: 20 000 строк.
    sample_size = min(20000, len(leningrad_df))
    docs_df = leningrad_df.sample(sample_size, random_state=42)
    # Очищаем короткие комментарии (меньше 3 слов) для качественного эмбеддинга
    docs = docs_df[docs_df['word_count'] > 3]['text'].tolist()
    timestamps = docs_df[docs_df['word_count'] > 3]['year'].tolist()

    print(f"Запуск BERTopic на {len(docs)} документах...")
    
    # Инициализируем модель с лемматизацией (через CountVectorizer для стоп-слов)
    # Используем nltk stopwords, дополненные специфичными для рунета мусорными словами
    import nltk
    from nltk.corpus import stopwords
    nltk.download('stopwords', quiet=True)
    stop_words = stopwords.words('russian') + ['это', 'как', 'так', 'что', 'за', 'из', 'от', 'все', 'кто', 'просто', 'очень', 'видео']
    
    vectorizer_model = CountVectorizer(stop_words=stop_words, ngram_range=(1, 2))
    
    # Легковесная модель SentenceTransformers
    embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    
    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        language="multilingual",
        calculate_probabilities=False,
        nr_topics=15 # Ограничиваем количество макро-тем
    )
    
    topics, probs = topic_model.fit_transform(docs)
    
    # Визуализация 1: Горизонтальный Bar Chart ключевых тем (без темы -1 "Шум")
    fig2 = topic_model.visualize_barchart(top_n_topics=8, n_words=6, width=300, height=300)
    fig2.update_layout(title="Ключевые семантические кластеры памяти (Ленинград)",
                       plot_bgcolor='white', paper_bgcolor='white', font_family="Arial")
    save_plot(fig2, "02_bertopic_clusters")
    
    # Визуализация 2: Динамика тем во времени (Эволюция нарратива)
    topics_over_time = topic_model.topics_over_time(docs, timestamps)
    fig2_time = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=5)
    fig2_time.update_layout(title="Эволюция тем во времени", plot_bgcolor='white', paper_bgcolor='white')
    save_plot(fig2_time, "02_bertopic_evolution")
else:
    print("Датасет leningrad пуст или не найден.")

--- Гипотеза 2: Семантическая структура памяти (BERTopic) ---
Запуск BERTopic на 18040 документах...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
print("--- Гипотеза 3: Морфология исторического дискурса (Глубина) ---")

# Группируем данные по годам (ограничим с 2012 года, когда YouTube стал массовым в РФ)
h3_df = df[df['year'] >= 2012].copy()

# Считаем количество корневых комментариев (parent_comment_id is NaN) и ответов
discussion_stats = h3_df.groupby('year').agg(
    total_comments=('text', 'count'),
    roots=('is_reply', lambda x: (x == 0).sum()),
    replies=('is_reply', lambda x: (x == 1).sum())
).reset_index()

# Коэффициент глубины: сколько ответов приходится на 1 корневой комментарий
discussion_stats['reply_to_root_ratio'] = discussion_stats['replies'] / discussion_stats['roots'].replace(0, 1)

# Создаем график с двумя осями: объем обсуждения и глубина
fig3 = make_subplots(specs=[[{"secondary_y": True}]])

# 1. Столбцы: Общий объем комментариев (фон)
fig3.add_trace(
    go.Bar(x=discussion_stats['year'], y=discussion_stats['total_comments'], 
           name='Объем комментариев', marker_color=ACADEMIC_PALETTE[5], opacity=0.4),
    secondary_y=False,
)

# 2. Линия: Коэффициент глубины дискуссии
fig3.add_trace(
    go.Scatter(x=discussion_stats['year'], y=discussion_stats['reply_to_root_ratio'], 
               name='Глубина дискуссии (Ответы / Корни)', 
               mode='lines+markers', line=dict(color=ACADEMIC_PALETTE[1], width=4),
               marker=dict(size=8)),
    secondary_y=True,
)

fig3.update_layout(
    title="Морфология дискурса: Переход от высказываний к столкновениям",
    xaxis_title="Год",
    plot_bgcolor='white', paper_bgcolor='white', font_family="Arial",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig3.update_yaxes(title_text="Абсолютный объем", secondary_y=False, showgrid=False)
fig3.update_yaxes(title_text="Коэффициент глубины", secondary_y=True, showgrid=True, gridcolor='lightgray')

# Отмечаем ключевые политические точки
fig3.add_vline(x=2014, line_dash="dot", line_color=ACADEMIC_PALETTE[0])
fig3.add_annotation(x=2014, y=0.05, yref="paper", text="2014", showarrow=False, font=dict(color=ACADEMIC_PALETTE[0], size=14))

fig3.add_vline(x=2022, line_dash="dot", line_color=ACADEMIC_PALETTE[0])
fig3.add_annotation(x=2022, y=0.05, yref="paper", text="2022", showarrow=False, font=dict(color=ACADEMIC_PALETTE[0], size=14))

save_plot(fig3, "03_discussion_depth_ratio")

--- Гипотеза 3: Морфология исторического дискурса (Глубина) ---


In [9]:
print("--- Подготовка: Индекс конфликтогенности (Harshness Index) ---")

# Расширенный академический словарь маркеров агрессии, радикализации и обсценной лексики
# Используем корни для покрытия словоформ
conflict_dict = [
    r'предател', r'тварь', r'мраз', r'ублюд', r'идиот', r'дебил', r'ложь', r'врань', 
    r'лицемер', r'позор', r'фашист', r'нацист', r'сволоч', r'вырод', r'гнид', r'скот', 
    r'рабск', r'террорист', r'оккупант', r'рашист', r'бандер', r'хохл', r'кацап', r'москал',
    r'орк\b', r'орки\b', r'свино', r'русофоб', r'кремлебот', r'либераст', r'ватник', r'совок'
]
conflict_regex = '|'.join(conflict_dict)

# Считаем индекс (кол-во конфликтогенных корней / на 100 слов текста)
df['conflict_hits'] = df['text'].str.count(conflict_regex)
df['conflict_index'] = (df['conflict_hits'] / df['word_count']) * 100

--- Подготовка: Индекс конфликтогенности (Harshness Index) ---


In [10]:
print("--- Гипотеза 4 и 5: Социальное одобрение конфликтогенности ---")

# Чтобы избежать искажений от мега-каналов, рассчитываем квантили лайков ВНУТРИ КАЖДОГО ВИДЕО
# Для ускорения считаем только для видео, где > 50 комментариев
video_counts = df['video_id'].value_counts()
valid_videos = video_counts[video_counts > 50].index

df_h4 = df[df['video_id'].isin(valid_videos)].copy()

# Функция ранжирования лайков внутри группы (видео)
df_h4['like_percentile'] = df_h4.groupby('video_id')['like_count'].rank(pct=True)

# Разбиваем на категории социальной поддержки
conditions = [
    df_h4['like_percentile'] >= 0.90, # Топ 10% самых залайканных
    (df_h4['like_percentile'] >= 0.50) & (df_h4['like_percentile'] < 0.90), # Выше среднего
    df_h4['like_percentile'] < 0.50 # Игнорируемые/Средние
]
choices = ['Топ-10% (Лидеры мнений)', 'Выше среднего', 'Фон (Без поддержки)']
df_h4['support_tier'] = np.select(conditions, choices, default='Фон (Без поддержки)')

--- Гипотеза 4 и 5: Социальное одобрение конфликтогенности ---


In [11]:
# График 4: Распределение агрессии по уровням поддержки
h4_data = df_h4.groupby('support_tier')['conflict_index'].mean().reindex(choices).reset_index()

fig4 = px.bar(h4_data, x='support_tier', y='conflict_index', 
              title="Гипотеза 4: Социальное одобрение резкой риторики",
              labels={'support_tier': 'Уровень поддержки аудиторией', 'conflict_index': 'Средний индекс конфликтогенности'},
              color='support_tier',
              color_discrete_sequence=[ACADEMIC_PALETTE[1], ACADEMIC_PALETTE[3], ACADEMIC_PALETTE[5]])
fig4.update_layout(showlegend=False)
save_plot(fig4, "04_social_approval_of_conflict")

In [12]:
# График 5: Изменение нормы поддержки во времени (Динамика лидеров мнений vs Фон)
h5_data = df_h4.groupby(['year', 'support_tier'])['conflict_index'].mean().reset_index()
h5_data = h5_data[h5_data['year'] >= 2012]

fig5 = px.line(h5_data, x='year', y='conflict_index', color='support_tier', markers=True,
               title="Гипотеза 5: Динамика радикализации (Эволюция нормы)",
               labels={'year': 'Год', 'conflict_index': 'Индекс конфликтогенности'},
               color_discrete_sequence=[ACADEMIC_PALETTE[1], ACADEMIC_PALETTE[3], ACADEMIC_PALETTE[5]])

fig5.add_vline(x=2014, line_dash="dash", line_color='gray')
fig5.add_vline(x=2022, line_dash="dash", line_color='gray')
save_plot(fig5, "05_evolution_of_tolerance")

In [14]:
import re

In [15]:
print("--- Гипотеза 6: Деконструкция вклада союзников (Оконный поиск) ---")

# Regex с использованием \W+(?:\w+\W+){0,7} означает "не более 7 слов между концептами"
# Это защищает нас от ложных срабатываний в длинных комментариях
ally_deval_pattern = re.compile(
    r'(?:\b(сша|америк|британи|англия|ленд-лиз|черчилль|рузвельт|союзники)\b'
    r'(?:\W+\w+){0,7}\W+'
    r'(тушенка|наживались|выгодно|поздно|платили|золотом|отсиживались|предатели|специально|ждали)\b)|'
    r'(?:\b(тушенка|наживались|выгодно|поздно|платили|золотом|отсиживались|предатели|специально|ждали)\b'
    r'(?:\W+\w+){0,7}\W+'
    r'(сша|америк|британи|англия|ленд-лиз|черчилль|рузвельт|союзники)\b)', 
    re.IGNORECASE
)

# Выполняем поиск (бинарный флаг)
df['devalues_allies_window'] = df['text'].apply(lambda x: 1 if ally_deval_pattern.search(x) else 0)

h6_data = df[df['year'] >= 2010].groupby('year')['devalues_allies_window'].mean().reset_index()
h6_data['devalues_allies_window'] *= 10000 # В базисных пунктах для наглядности (на 10 000 комментариев)

--- Гипотеза 6: Деконструкция вклада союзников (Оконный поиск) ---


TypeError: bar() got an unexpected keyword argument 'marker_color'

In [16]:
fig6 = px.bar(h6_data, x='year', y='devalues_allies_window',
              title="Гипотеза 6: Деконструкция вклада Союзников (Оконный поиск)",
              labels={'devalues_allies_window': 'Частота нарратива обесценивания (на 10к комм.)', 
                      'year': 'Год'},
              color_discrete_sequence=['blue'])

fig6.add_vline(x=2014, line_dash="dot", line_color=ACADEMIC_PALETTE[1])
fig6.add_vline(x=2022, line_dash="dot", line_color=ACADEMIC_PALETTE[1])
save_plot(fig6, "06_allies_devaluation_window")

In [17]:
print("--- Гипотеза 7: Конфликтогенность национальных локусов памяти ---")

# Словари этнонимов и топонимов (исторические и современные)
geo_entities = {
    'Украина': r'украин|хохл|киев|днепр|харьков|одесс|бандер|упа\b',
    'Беларусь': r'беларус|белорусс|минск|брест|хатынь',
    'Кавказ': r'кавказ|чечен|грузин|армян|азерба|дагестан|баку|тбилиси',
    'Ср. Азия / Казахстан': r'казах|панфилов|узбек|таджик|туркмен|киргиз|алмат|ташкент',
    'Европа (Критика)': r'европ|франц|польш|поляк|прибалтик|латви|литв|эстони'
}

geo_results = []
global_harshness = df['conflict_index'].mean()

for region, pattern in geo_entities.items():
    mask = df['text'].str.contains(pattern, case=False, regex=True)
    subset = df[mask]
    if len(subset) > 500: # Отсекаем стат. выбросы
        geo_results.append({
            'Макрорегион': region,
            'Индекс конфликтогенности': subset['conflict_index'].mean(),
            'Базовый фон': global_harshness
        })

df_geo = pd.DataFrame(geo_results)
df_geo_melt = df_geo.melt(id_vars='Макрорегион', var_name='Тип метрики', value_name='Значение')

fig7 = px.bar(df_geo_melt, x='Макрорегион', y='Значение', color='Тип метрики', barmode='group',
              title="Гипотеза 7: Конфликтогенность национальных локусов памяти",
              color_discrete_map={'Индекс конфликтогенности': ACADEMIC_PALETTE[1], 'Базовый фон': ACADEMIC_PALETTE[5]},
              labels={'Значение': 'Индекс (Агрессия / 100 слов)'})

fig7.update_layout(yaxis_range=[0, df_geo_melt['Значение'].max() * 1.2])
save_plot(fig7, "07_geo_conflict_loci")

--- Гипотеза 7: Конфликтогенность национальных локусов памяти ---


In [21]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from transformers import pipeline
from tqdm.auto import tqdm
import torch

print("--- Подготовка: Загрузка метаданных видео ---")

video_files = list(Path(".").rglob("data_videos_*.csv"))
if video_files:
    df_vid_list = []
    for f in video_files:
        try:
            # Читаем только нужные колонки для экономии оперативной памяти
            temp = pd.read_csv(f, usecols=lambda c: c in ['video_id', 'video', 'id_video', 'channel_id', 'publish_date', 'view_count'], low_memory=False)
            
            # Унификация ключа video_id
            for old in ['video', 'id_video']:
                if old in temp.columns: 
                    temp.rename(columns={old: 'video_id'}, inplace=True)
                    
            # Переименовываем дату публикации видео, чтобы она не слилась с датой комментария
            if 'publish_date' in temp.columns:
                temp.rename(columns={'publish_date': 'video_publish_date'}, inplace=True)
                
            df_vid_list.append(temp)
        except Exception as e:
            print(f"Ошибка чтения видео файла {f.name}: {e}")
            
    if df_vid_list:
        df_vid = pd.concat(df_vid_list, ignore_index=True).drop_duplicates('video_id')
        # Джойним метаданные к комментариям
        df = df.merge(df_vid, on='video_id', how='left')
        del df_vid_list, df_vid; gc.collect()
        print("Метаданные видео успешно подгружены.")
else:
    print("Файлы видео не найдены.")

# Расчет Time Lag (Инфраструктура старения дискурса)
if 'video_publish_date' in df.columns:
    df['video_publish_date'] = pd.to_datetime(df['video_publish_date'], errors='coerce')
    # Разница в днях между публикацией видео и оставленным комментарием
    df['time_lag_days'] = (df['comment_date'] - df['video_publish_date']).dt.days
    
    # Отсекаем отрицательные значения (ошибки парсинга дат)
    df = df[df['time_lag_days'] >= 0]
    
# Вычисляем Политизацию (для последующих гипотез)
politicized_markers = r'украин|росси|сша|нато|сво\b|путин|байден|зеленск|запад'
df['is_politicized'] = df['text'].str.contains(politicized_markers, case=False).astype(int)

--- Подготовка: Загрузка метаданных видео ---
Метаданные видео успешно подгружены.


In [24]:
# --- Гипотеза 8: Инфраструктура старения дискурса ---
print("--- Гипотеза 8: Расчет Time Lag ---")
bins = [-1, 7, 30, 180, 365, 1825, float('inf')]
labels = ['Первая неделя', 'Первый месяц', 'Полгода', 'Первый год', 'До 5 лет', 'Старше 5 лет']
df['lag_bin'] = pd.cut(df['time_lag_days'], bins=bins, labels=labels)
    
h8_data = df.groupby('lag_bin')['conflict_index'].mean().reset_index()
    
fig8 = px.line(h8_data, x='lag_bin', y='conflict_index', markers=True,
                title="Гипотеза 8: Эффект старения видео (Эхо-камеры старых тредов)",
                labels={'lag_bin': 'Время с момента публикации видео', 'conflict_index': 'Индекс конфликтогенности'},
                color_discrete_sequence=[ACADEMIC_PALETTE[2]])
fig8.update_traces(line=dict(width=3), marker=dict(size=10))
save_plot(fig8, "08_time_lag_conflict")

--- Гипотеза 8: Расчет Time Lag ---


In [23]:
print("--- Гипотеза 9: Тематические пузыри (Эхо-камеры по темам) ---")
h9_data = df.groupby('dataset')['conflict_index'].mean().sort_values().reset_index()

fig9 = px.bar(h9_data, x='conflict_index', y='dataset', orientation='h',
              title="Гипотеза 9: Топология конфликтогенности (Рейтинг исторических сюжетов)",
              labels={'conflict_index': 'Средний индекс агрессии', 'dataset': 'Исторический датасет'},
              color='conflict_index', color_continuous_scale='Reds')
save_plot(fig9, "09_dataset_echo_chambers")

--- Гипотеза 9: Тематические пузыри (Эхо-камеры по темам) ---


In [26]:
df.columns

Index(['text', 'comment_date', 'like_count', 'video_id', 'parent_comment_id',
       'dataset', 'year', 'month_year', 'is_reply', 'modern_hits',
       'word_count', 'actualization_index', 'quarter_year', 'conflict_hits',
       'conflict_index', 'devalues_allies_window', 'is_politicized',
       'view_count', 'video_publish_date', 'channel_id', 'time_lag_days',
       'lag_bin'],
      dtype='object')

In [27]:
import pandas as pd
import numpy as np
import gc
from pathlib import Path

def load_unified_dataset():
    print("--- Шаг 1: Загрузка метаданных ВИДЕО ---")
    video_files = list(Path(".").rglob("data_videos_*.csv"))
    df_vid_list = []
    
    for f in video_files:
        # Загружаем метаданные видео
        temp_v = pd.read_csv(f, low_memory=False, on_bad_lines='skip')
        # Унифицируем ID видео и канала
        rename_v = {
            'video': 'video_id', 'id_video': 'video_id',
            'channel_id': 'channel_id', 'id_channel': 'channel_id',
            'publish_date': 'video_publish_date'
        }
        temp_v.rename(columns={k: v for k, v in rename_v.items() if k in temp_v.columns}, inplace=True)
        # Оставляем только нужные метаданные
        cols_to_keep = ['video_id', 'channel_id', 'video_publish_date', 'view_count']
        df_vid_list.append(temp_v[[c for c in cols_to_keep if c in temp_v.columns]])

    df_vid = pd.concat(df_vid_list, ignore_index=True).drop_duplicates('video_id')
    print(f"Загружено метаданных для {len(df_vid)} уникальных видео.")

    print("\n--- Шаг 2: Загрузка КОММЕНТАРИЕВ ---")
    comment_files = list(Path(".").rglob("data_comments_*.csv"))
    df_comm_list = []
    
    for f in comment_files:
        temp_c = pd.read_csv(f, low_memory=False, on_bad_lines='skip')
        # Унифицируем колонки автора и даты
        rename_c = {
            'video': 'video_id', 'video_url': 'video_id', 'id_video': 'video_id',
            'author_id': 'author_id', 'commenter_channel_id': 'author_id', 'author': 'author_id',
            'publish_date': 'comment_date', 'comment_publish_date': 'comment_date',
            'text': 'text', 'like_count': 'like_count'
        }
        temp_c.rename(columns={k: v for k, v in rename_c.items() if k in temp_c.columns}, inplace=True)
        
        # Если author_id все еще нет, используем заглушку, чтобы код не падал, 
        # но в норме он должен подтянуться из колонок 'author...'
        if 'author_id' not in temp_c.columns:
            temp_c['author_id'] = 'unknown_user'
            
        temp_c['dataset'] = f.stem.replace("data_comments_", "")
        df_comm_list.append(temp_c)

    df = pd.concat(df_comm_list, ignore_index=True)
    print(f"Загружено {len(df)} комментариев.")

    print("\n--- Шаг 3: Слияние (Merge) ---")
    # Объединяем комментарии с метаданными их видео
    df = df.merge(df_vid, on='video_id', how='left')
    
    # Финальная очистка типов
    df['comment_date'] = pd.to_datetime(df['comment_date'], errors='coerce')
    df['video_publish_date'] = pd.to_datetime(df['video_publish_date'], errors='coerce')
    df = df.dropna(subset=['comment_date', 'text'])
    
    # Добавляем технические колонки для экспериментов
    df['year'] = df['comment_date'].dt.year
    df['word_count'] = df['text'].str.split().str.len().fillna(0).replace(0, 1)
    
    if 'video_publish_date' in df.columns:
        df['time_lag_days'] = (df['comment_date'] - df['video_publish_date']).dt.days
        df.loc[df['time_lag_days'] < 0, 'time_lag_days'] = np.nan # Убираем ошибки дат
    
    print("Датасет готов. Колонки:", df.columns.tolist())
    
    del df_vid, df_comm_list, df_vid_list; gc.collect()
    return df

# Перезаписываем глобальный df
df = load_unified_dataset()

--- Шаг 1: Загрузка метаданных ВИДЕО ---
Загружено метаданных для 44961 уникальных видео.

--- Шаг 2: Загрузка КОММЕНТАРИЕВ ---
Загружено 4825616 комментариев.

--- Шаг 3: Слияние (Merge) ---
Датасет готов. Колонки: ['Unnamed: 0', 'comment_id', 'text', 'comment_date', 'like_count', 'reply_count', 'video_id', 'author_id', 'parent_comment_id', 'dataset', 'channel_id', 'video_publish_date', 'view_count', 'year', 'word_count', 'time_lag_days']


In [29]:
import numpy as np
import pandas as pd
import gc

def calculate_all_metrics(df):
    print("--- Запуск массового расчета метрик (H8-H16) ---")
    
    # 0. Базовая подготовка текста
    df['text'] = df['text'].fillna('').astype(str).str.lower()
    df['word_count'] = df['text'].str.split().str.len().replace(0, 1)

    # 1. Индекс конфликтогенности (Harshness Index) - для H4, H5, H7, H8, H9, H11, H16
    print("Расчет индекса конфликтогенности...")
    conflict_words = [
        r'предател', r'тварь', r'мраз', r'ублюд', r'идиот', r'дебил', r'ложь', r'врань', 
        r'лицемер', r'позор', r'фашист', r'нацист', r'сволоч', r'вырод', r'гнид', r'скот', 
        r'рабск', r'террорист', r'оккупант', r'рашист', r'бандер', r'хохл', r'кацап', r'москал',
        r'орк\b', r'орки\b', r'свино', r'русофоб', r'кремлебот', r'либераст', r'ватник', r'совок'
    ]
    df['conflict_hits'] = df['text'].str.count('|'.join(conflict_words))
    df['conflict_index'] = (df['conflict_hits'] / df['word_count']) * 100

    # 2. Маркер Политизации - для H10
    print("Расчет маркера политизации...")
    politicized_markers = r'украин|росси|сша|нато|сво\b|путин|байден|зеленск|запад'
    df['is_politicized'] = df['text'].str.contains(politicized_markers, case=False).astype(int)

    # 3. Маркеры исторического фрейминга (PMI) - для H14
    print("Расчет маркеров фрейминга (Evil/West/UA/Ger)...")
    df['has_evil'] = df['text'].str.contains(r'фашист|нацист|нацизм|фашизм|рейх|гитлер', case=False).astype(int)
    df['has_west'] = df['text'].str.contains(r'запад|сша|америк|европ|байден|нато', case=False).astype(int)
    df['has_ua'] = df['text'].str.contains(r'украин|хохл|бандер|зеленск', case=False).astype(int)
    df['has_ger'] = df['text'].str.contains(r'герман|немец|немцы', case=False).astype(int)

    # 4. Индекс исторической глубины - для H16
    print("Расчет индекса исторической глубины...")
    history_markers = r'\b(194[1-5]|сталин|жуков|рокоссовский|гитлер|вермахт|ссср|власов|ленинград)\b'
    df['history_hits'] = df['text'].str.count(history_markers)
    df['historical_depth_index'] = (df['history_hits'] / df['word_count']) * 100

    # 5. Индекс структурной экспрессивности (Синтаксис) - для H13
    print("Расчет индекса экспрессивности...")
    # Считаем восклицания и вопросы (2 и более подряд)
    df['punct_express'] = df['text'].str.count(r'[!?!]{2,}')
    df['expressiveness_index'] = (df['punct_express'] / df['word_count']) * 100

    # 6. Вспомогательные флаги - для H3, H11
    df['is_reply'] = df['parent_comment_id'].notna().astype(int)
    df['is_conflict'] = (df['conflict_index'] > 0).astype(int)

    # 7. Тиры каналов (Scale) - для H10
    # Проверяем, есть ли channel_id. Если нет, берем video_id как прокси
    target_col = 'channel_id' if 'channel_id' in df.columns else 'video_id'
    print(f"Классификация масштаба каналов (по {target_col})...")
    channel_sizes = df.groupby(target_col)['video_id'].transform('nunique')
    
    # Используем pd.qcut для разбивки на 3 группы по охвату
    try:
        df['channel_tier'] = pd.qcut(channel_sizes, q=[0, 0.5, 0.9, 1.0], 
                                     labels=['Нишевые', 'Средние', 'Мега-каналы'], duplicates='drop')
    except:
        df['channel_tier'] = 'Средние' # Фолбэк если данных мало

    print("--- Расчет завершен. Память очищена. ---")
    gc.collect()
    return df

# Применяем функцию к нашему основному датафрейму
df = calculate_all_metrics(df)

# Проверка: выведем список колонок, чтобы убедиться в наличии 'is_politicized'
print("\nТекущие колонки датасета:", df.columns.tolist())

--- Запуск массового расчета метрик (H8-H16) ---
Расчет индекса конфликтогенности...
Расчет маркера политизации...
Расчет маркеров фрейминга (Evil/West/UA/Ger)...
Расчет индекса исторической глубины...
Расчет индекса экспрессивности...
Классификация масштаба каналов (по channel_id)...
--- Расчет завершен. Память очищена. ---

Текущие колонки датасета: ['Unnamed: 0', 'comment_id', 'text', 'comment_date', 'like_count', 'reply_count', 'video_id', 'author_id', 'parent_comment_id', 'dataset', 'channel_id', 'video_publish_date', 'view_count', 'year', 'word_count', 'time_lag_days', 'channel_tier', 'conflict_hits', 'conflict_index', 'is_politicized', 'has_evil', 'has_west', 'has_ua', 'has_ger', 'history_hits', 'historical_depth_index', 'punct_express', 'expressiveness_index', 'is_reply', 'is_conflict']


In [30]:
print("--- Гипотеза 10: Масштаб канала vs Политизация ---")
# Эмуляция тиров каналов по количеству комментариев (как прокси размера), если нет view_count
channel_activity = df.groupby('author_id')['text'].count() # Используем author_id видео или channel_id
# Допустим, channel_id есть. Если нет, этот шаг адаптируем под ваши колонки.
if 'channel_id' in df.columns:
    channel_sizes = df.groupby('channel_id')['text'].count()
    df['channel_tier'] = df['channel_id'].map(
        pd.qcut(channel_sizes, q=[0, 0.5, 0.9, 1.0], labels=['Нишевые', 'Средние', 'Мега-каналы'])
    )
    h10_data = df.groupby('channel_tier')['is_politicized'].mean().reset_index()
    h10_data['is_politicized'] *= 100 # В проценты
    
    fig10 = px.bar(h10_data, x='channel_tier', y='is_politicized',
                   title="Гипотеза 10: Степень политизации истории от масштаба канала",
                   labels={'channel_tier': 'Размер канала', 'is_politicized': 'Доля политизированных комментариев (%)'},
                   color_discrete_sequence=[ACADEMIC_PALETTE[3]])
    save_plot(fig10, "10_channel_scale_politicization")

--- Гипотеза 10: Масштаб канала vs Политизация ---


In [32]:
print("--- Гипотеза 11: Морфология участников дискурса (Исправленная) ---")

# 1. Считаем активность пользователей
user_activity = df.groupby('author_id').size().reset_index(name='comment_count')

# Проверяем квантили (если данных мало, они могут совпасть)
q80 = user_activity['comment_count'].quantile(0.80)
q95 = user_activity['comment_count'].quantile(0.95)

# 2. Определяем сегменты
conditions = [
    (user_activity['comment_count'] >= q95),
    (user_activity['comment_count'] >= q80) & (user_activity['comment_count'] < q95),
    (user_activity['comment_count'] < q80)
]
choices = ['Ядро (Топ-5% активных)', 'Регулярные (15%)', 'Периферия (Случайные)']

# Исправление: добавляем default строку, чтобы не было конфликта типов (str vs int)
user_activity['user_segment'] = np.select(conditions, choices, default='Периферия (Случайные)')

# 3. Мерджим обратно. 
# Перед мерджем удаляем старый user_segment, если он уже был в df (защита от повторного запуска)
if 'user_segment' in df.columns:
    df = df.drop(columns=['user_segment'])

df = df.merge(user_activity[['author_id', 'user_segment']], on='author_id', how='left')

# 4. Определяем конфликтный комментарий (если расчет метрик был запущен ранее)
if 'conflict_index' not in df.columns:
    # Краткий фолбэк расчет, если метрики не считали
    conflict_dict = r'предател|мраз|идиот|дебил|ложь|позор|фашист|нацист'
    df['conflict_index'] = (df['text'].str.count(conflict_dict) / df['word_count']) * 100

df['is_conflict'] = (df['conflict_index'] > 0).astype(int)

# 5. Агрегация данных для визуализации
h11_data = df.groupby('user_segment', observed=True).agg(
    total_users=('author_id', 'nunique'),
    total_comments=('text', 'count'),
    conflict_comments=('is_conflict', 'sum')
).reset_index()

# Считаем долю агрессии, которую вносит каждый сегмент в общий котел
h11_data['share_of_conflict'] = h11_data['conflict_comments'] / h11_data['conflict_comments'].sum() * 100

# 6. Визуализация
fig11 = px.pie(h11_data, values='share_of_conflict', names='user_segment', hole=0.4,
               title="Гипотеза 11: Кто генерирует агрессию? (Вклад сегментов пользователей)",
               color_discrete_sequence=[ACADEMIC_PALETTE[1], ACADEMIC_PALETTE[3], ACADEMIC_PALETTE[5]])

fig11.update_traces(textinfo='percent+label', marker=dict(line=dict(color='#000000', width=1)))
fig11.update_layout(font_family="Arial", title_x=0.5)

save_plot(fig11, "11_user_morphology")

# Печатаем статистику для проверки
print(h11_data[['user_segment', 'total_users', 'share_of_conflict']].to_string(index=False))

--- Гипотеза 11: Морфология участников дискурса (Исправленная) ---


          user_segment  total_users  share_of_conflict
 Периферия (Случайные)       948020          23.539589
      Регулярные (15%)       204252          24.481173
Ядро (Топ-5% активных)        66883          51.979238


In [33]:
print("--- Гипотеза 12: Историческая терапия (Zero-Shot Окно Сент-Окт 2022) ---")
# Выделяем окно стресса
stress_window = df[(df['comment_date'] >= '2022-08-01') & (df['comment_date'] <= '2022-11-30') & (df['word_count'] > 5)]

if len(stress_window) > 0:
    # Стратифицированный сэмпл: 2000 строк для быстрого прогона на T4
    sample_size = min(2000, len(stress_window))
    h12_sample = stress_window.sample(sample_size, random_state=42).copy()
    
    device = 0 if torch.cuda.is_available() else -1
    classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=device)
    
    # Академические классы
    labels = [
        "Апелляция к историческому триумфу и консолидация",
        "Тревожность, критика и исторический фатализм",
        "Нейтральная академическая память"
    ]
    
    print(f"Запуск нейросети на {len(h12_sample)} текстах (Ожидайте пару минут)...")
    
    # Batch processing
    class_results = []
    texts = h12_sample['text'].tolist()
    
    # Обработка батчами
    for out in tqdm(classifier(texts, labels, batch_size=16, multi_label=False), total=len(texts)):
        class_results.append(out['labels'][0])
        
    h12_sample['dominant_narrative'] = class_results
    
    # Агрегируем по неделям
    h12_sample['week'] = h12_sample['comment_date'].dt.to_period('W').dt.to_timestamp()
    h12_trend = h12_sample.groupby(['week', 'dominant_narrative']).size().unstack(fill_value=0)
    h12_trend_pct = h12_trend.div(h12_trend.sum(axis=1), axis=0) * 100
    h12_melt = h12_trend_pct.reset_index().melt(id_vars='week', var_name='Нарратив', value_name='Доля (%)')
    
    fig12 = px.line(h12_melt, x='week', y='Доля (%)', color='Нарратив', markers=True,
                    title="Гипотеза 12: Историческая терапия в период социального шока (Осень 2022)",
                    color_discrete_sequence=[ACADEMIC_PALETTE[4], ACADEMIC_PALETTE[1], ACADEMIC_PALETTE[5]])
    fig12.add_vline(x="2022-09-21", line_dash="dash", line_color='black')
    fig12.add_annotation(x="2022-09-21", y=90, text="Объявление мобилизации", showarrow=False)
    save_plot(fig12, "12_historical_therapy_zeroshot")

--- Гипотеза 12: Историческая терапия (Zero-Shot Окно Сент-Окт 2022) ---


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Запуск нейросети на 2000 текстах (Ожидайте пару минут)...


  0%|          | 0/2000 [00:00<?, ?it/s]

In [34]:
print("--- Гипотеза 13: Индекс структурной экспрессивности ---")
# Считаем слова капсом (минимум 3 буквы подряд в верхнем регистре)
# Важно: берем исходный датафрейм df до lower() (если вы делали lower при загрузке, капс утерян)
# Допустим, мы извлекаем экспрессивность из сырого текста, либо по знакам препинания:
df['punct_express'] = df['text'].str.count(r'[!?!]{2,}') # 2 и более знаков подряд
df['expressiveness_index'] = (df['punct_express'] / df['word_count']) * 100

h13_data = df[df['year'] >= 2010].groupby('year')['expressiveness_index'].mean().reset_index()

fig13 = px.line(h13_data, x='year', y='expressiveness_index', markers=True,
                title="Гипотеза 13: Структурная экспрессивность (Синтаксис напряжения)",
                labels={'year': 'Год', 'expressiveness_index': 'Индекс синтаксического надрыва'},
                color_discrete_sequence=[ACADEMIC_PALETTE[6]])
fig13.update_traces(line=dict(width=3))
save_plot(fig13, "13_expressiveness_syntax")

--- Гипотеза 13: Индекс структурной экспрессивности ---


In [35]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords
import re

print("--- Гипотеза 14: Смещение исторического фрейминга (PMI Matrix) ---")

# Определяем паттерны
evil_pattern = r'фашист|нацист|нацизм|фашизм|рейх|гитлер'
modern_west_pattern = r'запад|сша|америк|европ|байден|нато'
modern_ua_pattern = r'украин|хохл|бандер|зеленск'
history_ger_pattern = r'герман|немец|немцы'

# Считаем бинарные флаги наличия концептов
df['has_evil'] = df['text'].str.contains(evil_pattern, case=False).astype(int)
df['has_west'] = df['text'].str.contains(modern_west_pattern, case=False).astype(int)
df['has_ua'] = df['text'].str.contains(modern_ua_pattern, case=False).astype(int)
df['has_ger'] = df['text'].str.contains(history_ger_pattern, case=False).astype(int)

pmi_results = []
years = sorted(df[df['year'] >= 2012]['year'].unique())

for y in years:
    sub = df[df['year'] == y]
    N = len(sub)
    if N < 1000: continue
    
    P_evil = sub['has_evil'].sum() / N
    
    for actor, col, name in [('Запад/США', 'has_west', 'Запад'), 
                             ('Украина', 'has_ua', 'Украина'), 
                             ('Германия (Истор.)', 'has_ger', 'Германия')]:
        
        P_actor = sub[col].sum() / N
        P_both = len(sub[(sub['has_evil'] == 1) & (sub[col] == 1)]) / N
        
        # Защита от логарифма нуля: используем сглаживание Лапласа
        if P_both > 0 and P_actor > 0 and P_evil > 0:
            pmi = np.log2(P_both / (P_actor * P_evil))
        else:
            pmi = 0
            
        pmi_results.append({'Год': y, 'Актор': name, 'PMI': pmi})

df_pmi = pd.DataFrame(pmi_results)

fig14 = px.line(df_pmi, x='Год', y='PMI', color='Актор', markers=True,
                title="Гипотеза 14: Эволюция фрейминга 'Фашизм' (Pointwise Mutual Information)",
                labels={'PMI': 'PMI (Степень семантической связи)'},
                color_discrete_sequence=[ACADEMIC_PALETTE[2], ACADEMIC_PALETTE[1], ACADEMIC_PALETTE[5]])

fig14.add_hline(y=0, line_dash="dot", annotation_text="Отсутствие связи", annotation_position="bottom right")
fig14.add_vline(x=2014, line_dash="dash", line_color='gray')
fig14.add_vline(x=2022, line_dash="dash", line_color='gray')
save_plot(fig14, "14_semantic_framing_pmi")

--- Гипотеза 14: Смещение исторического фрейминга (PMI Matrix) ---


In [38]:
print("--- Гипотеза 15: Лингвистическая дивергенция (Log-Odds Ratio) ---")

# Формируем два полярных корпуса на основе маркеров политической риторики
loyal_pattern = r'наши\b|герои|защитники|денацификац|гойда'
critic_pattern = r'орки\b|рашист|агрессор|оккупант|бункер'

# Используем только политизированные длинные комментарии для чистоты
pool = df[(df['is_politicized'] == 1) & (df['word_count'] > 5)].copy()

# Выделяем группы (строго эксклюзивные)
pool['is_loyal'] = (pool['text'].str.contains(loyal_pattern) & ~pool['text'].str.contains(critic_pattern)).astype(int)
pool['is_critic'] = (pool['text'].str.contains(critic_pattern) & ~pool['text'].str.contains(loyal_pattern)).astype(int)

corp_loyal = pool[pool['is_loyal'] == 1]
corp_critic = pool[pool['is_critic'] == 1]

if len(corp_loyal) > 500 and len(corp_critic) > 500:
    # Сэмплируем для баланса и скорости (по 10к из каждого корпуса)
    S = min(10000, len(corp_loyal), len(corp_critic))
    texts_loyal = corp_loyal.sample(S, random_state=42)['text'].tolist()
    texts_critic = corp_critic.sample(S, random_state=42)['text'].tolist()
    
    stop_rus = stopwords.words('russian') + ['это', 'что', 'как', 'так', 'просто', 'очень', 'все', 'кто']
    vec = CountVectorizer(stop_words=stop_rus, min_df=5, max_df=0.5)
    
    # Обучаем словарь на объединенном корпусе
    X_all = vec.fit_transform(texts_loyal + texts_critic)
    vocab = vec.get_feature_names_out()
    
    # Считаем частоты (alpha = 1 для сглаживания)
    freq_loyal = np.array(X_all[:S].sum(axis=0)).flatten() + 1
    freq_critic = np.array(X_all[S:].sum(axis=0)).flatten() + 1
    
    N_loyal = freq_loyal.sum()
    N_critic = freq_critic.sum()
    
    # Расчет Log-Odds
    log_odds = np.log(freq_loyal / (N_loyal - freq_loyal)) - np.log(freq_critic / (N_critic - freq_critic))
    
    # Собираем датафрейм
    df_odds = pd.DataFrame({'word': vocab, 'log_odds': log_odds})
    
    # Топ-15 слов для каждого вектора
    top_loyal = df_odds.sort_values('log_odds', ascending=False).head(15)
    top_loyal['vector'] = 'Официально-патриотический дискурс'
    
    top_critic = df_odds.sort_values('log_odds', ascending=True).head(15)
    # Инвертируем значение для красивого отображения на графике
    top_critic['log_odds'] = np.abs(top_critic['log_odds'])
    top_critic['vector'] = 'Антисистемный дискурс'
    
    df_divergence = pd.concat([top_loyal, top_critic])
    
    fig15 = px.bar(df_divergence, x='log_odds', y='word', color='vector', facet_col='vector',
                   orientation='h', title="Гипотеза 15: Лингвистическая дивергенция (Log-Odds Ratio)",
                   color_discrete_sequence=[ACADEMIC_PALETTE[1], ACADEMIC_PALETTE[3]])
    
    fig15.update_yaxes(matches=None, showticklabels=True)
    save_plot(fig15, "15_linguistic_divergence")
else:
    print("Недостаточно данных для формирования полярных корпусов.")

--- Гипотеза 15: Лингвистическая дивергенция (Log-Odds Ratio) ---


In [37]:
print("--- Гипотеза 16: Ландшафт исторической памяти (Memory Landscape) ---")

# Индекс исторической плотности: наличие точных дат (1941, 1945), имен исторических деятелей
history_markers = r'\b(194[1-5]|сталин|жуков|рокоссовский|гитлер|вермахт|ссср|власов|ленинград)\b'
df['history_hits'] = df['text'].str.count(history_markers)
df['historical_depth_index'] = (df['history_hits'] / df['word_count']) * 100

# Если conflict_index не рассчитан в этой сессии, пересчитаем
if 'conflict_index' not in df.columns:
    conflict_dict = r'предател|мраз|идиот|дебил|ложь|позор|фашист|нацист|рашист|бандер|орк\b'
    df['conflict_index'] = (df['text'].str.count(conflict_dict) / df['word_count']) * 100

# Агрегация по датасетам (событиям)
h16_data = df.groupby('dataset').agg(
    historical_depth=('historical_depth_index', 'mean'),
    conflict_lvl=('conflict_index', 'mean'),
    volume=('text', 'count')
).reset_index()

# Фильтрация микро-датасетов (для чистоты графика)
h16_data = h16_data[h16_data['volume'] > 1000]

fig16 = px.scatter(h16_data, x='historical_depth', y='conflict_lvl', 
                   size='volume', color='conflict_lvl', text='dataset',
                   title="Гипотеза 16: Ландшафт памяти (Академичность vs Идеологизация)",
                   labels={
                       'historical_depth': 'Глубина исторического контекста (Индекс)', 
                       'conflict_lvl': 'Уровень политизированной агрессии (Индекс)',
                       'dataset': 'Тема'
                   },
                   color_continuous_scale='Magma', size_max=40)

fig16.update_traces(textposition='top center')

# Добавление квадрантов (медианные линии)
med_x = h16_data['historical_depth'].median()
med_y = h16_data['conflict_lvl'].median()
fig16.add_vline(x=med_x, line_dash="dot", line_color="gray", opacity=0.5)
fig16.add_hline(y=med_y, line_dash="dot", line_color="gray", opacity=0.5)

# Аннотации квадрантов
fig16.add_annotation(x=h16_data['historical_depth'].min(), y=h16_data['conflict_lvl'].max(),
                     text="Зона Холиваров", showarrow=False, font=dict(color='gray'), xanchor='left')
fig16.add_annotation(x=h16_data['historical_depth'].max(), y=h16_data['conflict_lvl'].min(),
                     text="Академическая Гавань", showarrow=False, font=dict(color='gray'), xanchor='right')

save_plot(fig16, "16_memory_landscape")

print("Все гипотезы (14-16) успешно рассчитаны и сохранены.")

--- Гипотеза 16: Ландшафт исторической памяти (Memory Landscape) ---


Все гипотезы (14-16) успешно рассчитаны и сохранены.
